# 📦 DataCo Supply Chain Ingestion Pipeline: **Bronze Layer**
---

### 📌 Architecture Overview
This notebook implements the **Bronze Layer Ingestion** of raw supply chain transactional logs from **Amazon S3** into a **Delta Lake Table** on Databricks.

* **Source File:** `DataCoSupplyChainDataset.csv` (~91.5 MB, 180k+ records, 53 columns)
* **Target Table:** `workspace.dataco_bronze.supply_chain_raw` (Delta Lake)

```
 +-----------------------+      +---------------------------+      +-------------------------------+
 |   Amazon S3 Bucket    |      |    Databricks Workspace   |      |    Bronze Layer (Delta Lake)  |
 |                       |      |                           |      |                               |
 |  Raw DataCo CSV File  | ---> |   Spark Schema Validation | ---> |      supply_chain_raw         |
 |   (unstructured S3)   |      |   & Audit Metadata Added  |      |   (optimized Delta format)    |
 +-----------------------+      +---------------------------+      +-------------------------------+
```

---
### ⚙️ Implementation Steps
1. **Configuration Widgets:** Sets up dropdowns and textboxes to configure the pipeline dynamically.
2. **Unity Catalog Authorization:** Accesses Amazon S3 securely via Databricks Unity Catalog, removing the need for hardcoded passwords/keys.
3. **Schema Enforcement:** Defines the table columns and data types explicitly using `StructType` to ensure data integrity and avoid slow inference scans.
4. **Lineage Ingestion Timestamp:** Adds a clean ingestion timestamp column (`ingested_at`) to keep track of when data was loaded.
5. **Delta Optimization:** Compacts the storage files and Z-Orders on `order_date` to make downstream queries extremely fast.


## 🛠️ Step 1: Configuration & Parameterization
Define widgets to pass parameters like S3 path and catalog name dynamically. Decoupling configuration from code is a standard data engineering practice.


In [0]:
# Define widgets with defaults matching your S3 and Databricks settings
dbutils.widgets.text("s3_bucket_uri", "s3a://dataco-analysis/", "S3 Bucket Path (URI)")
dbutils.widgets.text("s3_file_name", "DataCoSupplyChainDataset.csv", "S3 CSV File Name")
dbutils.widgets.text("target_catalog", "workspace", "Target Catalog")
dbutils.widgets.text("target_schema", "dataco_bronze", "Target Schema / Database")
dbutils.widgets.text("target_table", "supply_chain_raw", "Target Table Name")
dbutils.widgets.dropdown("write_mode", "overwrite", ["overwrite", "append"], "Write Mode")
dbutils.widgets.dropdown("sanitize_columns", "True", ["True", "False"], "Sanitize Columns to snake_case")


## 📖 Step 2: Read Parameter Values
Load parameters selected in the widgets above.


In [0]:
# Retrieve parameter values from widgets
s3_bucket_uri = dbutils.widgets.get("s3_bucket_uri").strip("/") + "/"
s3_file_name = dbutils.widgets.get("s3_file_name")
s3_full_path = f"{s3_bucket_uri}{s3_file_name}"
target_catalog = dbutils.widgets.get("target_catalog").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
target_table = dbutils.widgets.get("target_table").strip()
write_mode = dbutils.widgets.get("write_mode").strip()
sanitize_columns = dbutils.widgets.get("sanitize_columns") == "True"

print(f"Ingesting from S3 path: {s3_full_path}")


## 📝 Step 3: Explicit Schema Definition
Define the explicit structure (`StructType`) of the 53 columns to ensure columns like zip codes are read as text to keep leading zeros, and numerical types are set correctly.


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define explicit PySpark schema for the 53 columns
raw_supply_chain_schema = StructType([
    StructField("Type", StringType(), True),
    StructField("Days for shipping (real)", IntegerType(), True),
    StructField("Days for shipment (scheduled)", IntegerType(), True),
    StructField("Benefit per order", DoubleType(), True),
    StructField("Sales per customer", DoubleType(), True),
    StructField("Delivery Status", StringType(), True),
    StructField("Late_delivery_risk", IntegerType(), True),
    StructField("Category Id", IntegerType(), True),
    StructField("Category Name", StringType(), True),
    StructField("Customer City", StringType(), True),
    StructField("Customer Country", StringType(), True),
    StructField("Customer Email", StringType(), True),
    StructField("Customer Fname", StringType(), True),
    StructField("Customer Id", IntegerType(), True),
    StructField("Customer Lname", StringType(), True),
    StructField("Customer Password", StringType(), True),
    StructField("Customer Segment", StringType(), True),
    StructField("Customer State", StringType(), True),
    StructField("Customer Street", StringType(), True),
    StructField("Customer Zipcode", StringType(), True),  # Keeps leading zeros
    StructField("Department Id", IntegerType(), True),
    StructField("Department Name", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Market", StringType(), True),
    StructField("Order City", StringType(), True),
    StructField("Order Country", StringType(), True),
    StructField("Order Customer Id", IntegerType(), True),
    StructField("order date (DateOrders)", StringType(), True), # Read as text, parsed later
    StructField("Order Id", IntegerType(), True),
    StructField("Order Item Cardprod Id", IntegerType(), True),
    StructField("Order Item Discount", DoubleType(), True),
    StructField("Order Item Discount Rate", DoubleType(), True),
    StructField("Order Item Id", IntegerType(), True),
    StructField("Order Item Product Price", DoubleType(), True),
    StructField("Order Item Profit Ratio", DoubleType(), True),
    StructField("Order Item Quantity", IntegerType(), True),
    StructField("Sales", DoubleType(), True),
    StructField("Order Item Total", DoubleType(), True),
    StructField("Order Profit Per Order", DoubleType(), True),
    StructField("Order Region", StringType(), True),
    StructField("Order State", StringType(), True),
    StructField("Order Status", StringType(), True),
    StructField("Order Zipcode", StringType(), True),
    StructField("Product Card Id", IntegerType(), True),
    StructField("Product Category Id", IntegerType(), True),
    StructField("Product Description", StringType(), True),
    StructField("Product Image", StringType(), True),
    StructField("Product Name", StringType(), True),
    StructField("Product Price", DoubleType(), True),
    StructField("Product Status", IntegerType(), True),
    StructField("shipping date (DateOrders)", StringType(), True), # Read as text, parsed later
    StructField("Shipping Mode", StringType(), True)
])


## 📥 Step 4: CSV Data Load
Reads the raw CSV file from S3 using Unity Catalog's authorization. 
* **Encoding:** `ISO-8859-1` (Latin-1) to safely read accented characters.
* **multiLine:** `true` to allow description fields with line breaks to parse without breaking rows.


In [0]:
# Read S3 file directly (Unity Catalog external storage handles authentication)
df_raw = (spark.read
    .format("csv")
    .option("header", "true")
    .option("encoding", "ISO-8859-1")
    .option("multiLine", "true")
    .schema(raw_supply_chain_schema)
    .load(s3_full_path)
)

print(f"✅ Successfully loaded: {df_raw.count():,} records.")


## 🧹 Step 5: Column Sanitization & Date Parsing
Standardizes raw headers to clean `snake_case` (e.g. `Days for shipping (real)` becomes `days_for_shipping_real`) and casts date text fields into Spark Timestamps.


In [0]:
import re
from pyspark.sql.functions import to_timestamp, col

def clean_column_name(col_name: str) -> str:
    clean = col_name.strip().lower()
    clean = clean.replace(" (real)", "_real")
    clean = clean.replace(" (scheduled)", "_scheduled")
    clean = clean.replace(" (dateorders)", "")
    clean = re.sub(r'[^a-zA-Z0-9_]', '_', clean)
    clean = re.sub(r'_+', '_', clean)
    return clean.strip('_')

# Apply header transformation if enabled
if sanitize_columns:
    sanitized_cols = [clean_column_name(c) for c in df_raw.columns]
    df_transformed = df_raw.toDF(*sanitized_cols)
    print("✅ Column names sanitized to snake_case.")
else:
    df_transformed = df_raw
    print("ℹ️ Column name sanitization skipped.")

# Resolve target date column names
order_date_col = "order_date" if sanitize_columns else "order date (DateOrders)"
shipping_date_col = "shipping_date" if sanitize_columns else "shipping date (DateOrders)"

# Parse date strings into structured Timestamps
csv_timestamp_format = "M/d/yyyy H:m"
df_transformed = df_transformed \
    .withColumn(order_date_col, to_timestamp(order_date_col, csv_timestamp_format)) \
    .withColumn(shipping_date_col, to_timestamp(shipping_date_col, csv_timestamp_format))

print("✅ Date string columns parsed to Timestamps successfully.")


## 🛡️ Step 6: Add Ingestion Timestamp
Appends a processing timestamp to record when the raw data was ingested. We avoid adding user email details or S3 file paths here to ensure the notebook is safe to share on public GitHub repositories.


In [0]:
from pyspark.sql.functions import current_timestamp

# Add ingestion timestamp
df_bronze = df_transformed.withColumn("ingested_at", current_timestamp())
print("✅ Added ingested_at timestamp column.")


## 💾 Step 7: Write to Delta Table
Saves the ingested DataFrame as a structured Delta table inside your catalog schema.


In [0]:
# Resolve target catalog and schema database paths
if target_catalog and target_catalog.lower() != "hive_metastore":
    full_table_path = f"`{target_catalog}`.`{target_schema}`.`{target_table}`"
    spark.sql(f"CREATE DATABASE IF NOT EXISTS `{target_catalog}`.`{target_schema}`")
else:
    full_table_path = f"`{target_schema}`.`{target_table}`"
    spark.sql(f"CREATE DATABASE IF NOT EXISTS `{target_schema}`")

print(f"💾 Saving Delta Table to: {full_table_path}")

# Write target table
(df_bronze.write
    .format("delta")
    .mode(write_mode)
    .option("mergeSchema", "true")
    .saveAsTable(full_table_path)
)

print(f"🚀 Delta table load completed successfully in {write_mode.upper()} mode.")


## ⚡ Step 8: Delta Storage Optimization
Runs table compaction (`OPTIMIZE`) and physically Z-Orders date fields to make date-range queries much faster in reporting dashboards.


In [0]:
# Optimize storage layout
optimize_query = f"OPTIMIZE {full_table_path} ZORDER BY ({order_date_col})"
print(f"⚡ Executing Compaction: {optimize_query}")
spark.sql(optimize_query)
print("⚡ Delta storage optimization complete.")


## 🔎 Step 9: Ingestion Validation
Performs a quick validation check by displaying a preview of the ingested Delta table records.


In [0]:
# Print total rows in Delta catalog table
df_validation = spark.read.table(full_table_path)
print(f"📊 Total records in Bronze Delta table: {df_validation.count():,}")

# Preview records
display(df_validation.limit(5))
